<a href="https://colab.research.google.com/github/AmrBr/Reverse-Dictionary/blob/main/Reverse_Dictionary.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**INSTALLS**

In [ ]:
!pip install camel-tools

**CONSTANTS**

In [1]:
SEED = 42
SPLITS = ["train", "validation", "test"]
COLUMNS_TO_REMOVE = ['id', 'pos', 'electra', 'bertseg', 'bertmsa']
WORD = 'word'
GLOSS = 'gloss'
DS1_PATH = 'Abreekaa/Arabic-Reverse-Dictionary'
DS2_PATH = 'riotu-lab/arabic_reverse_dictionary'
CLEAN_DATA_HF_PATH = 'Abreekaa/arabic-reverse-dictionary-merged'

## Data Preprocessing



In [ ]:
from datasets import load_dataset

def load_data():
  dataset_1 = load_dataset(DS1_PATH)
  dataset_bulk = load_dataset(DS2_PATH)

  dataset_bulk = dataset_bulk['train']
  return dataset_1, dataset_bulk

In [ ]:
def remove_dublicates(ds1, ds2):
  seen = set()

  for split in SPLITS:
      for ex in ds1[split]:
          seen.add((ex[WORD], ex[GLOSS]))

  ds_bulk_clean = ds2.filter(lambda ex: not_duplicate(ex, seen))
  return ds_bulk_clean

def not_duplicate(ex, seen):
    return (ex[WORD], ex[GLOSS]) not in seen

def deduplicate(ds):
    seen = set()

    def is_unique(ex):
        key = (ex[WORD], ex[GLOSS])
        if key in seen:
            return False
        seen.add(key)
        return True

    return ds.filter(is_unique)

In [ ]:
from camel_tools.utils.dediac import dediac_ar
from camel_tools.utils.normalize import normalize_alef_maksura_ar, normalize_alef_ar, normalize_teh_marbuta_ar

def dediacritize(ex):
  return dediac_ar(ex)

def remove_tatweel(ex):
  return ex.replace("ـ", "")

def normalize_text(ex):
  ex = normalize_alef_maksura_ar(ex)
  ex = normalize_alef_ar(ex)
  ex = normalize_teh_marbuta_ar(ex)
  return ex

In [ ]:
def clean_text(text):
    if text is None:
        return ""

    text = text.strip()
    if not text:
        return ""

    text = remove_tatweel(text)
    text = normalize_text(text)
    text = dediacritize(text)
    return text

In [ ]:
def preprocess_text(ex):
    return {
        WORD: clean_text(ex[WORD]),
        GLOSS: clean_text(ex[GLOSS]),
    }

In [ ]:
def preprocess_dataset(ds):
    ds = ds.map(preprocess_text)
    empty_count = sum(1 for ex in ds if (ex[WORD] == "" and ex[GLOSS] == ""))
    print("Empty examples: {}".format(empty_count))
    ds = ds.filter(
        lambda ex: ex[WORD] != "" and ex[GLOSS] != ""
    )
    return ds

In [ ]:
from datasets import concatenate_datasets

def merge_datasets(ds):
  return concatenate_datasets(ds)

In [ ]:
def analyze_dataset_integrity(ds, WORD_KEY='word', GLOSS_KEY='gloss'):
    splits = ['train', 'validation', 'test']

    print(f"{'Metric':<25} | {'Train':<10} | {'Val':<10} | {'Test':<10}")
    print("-" * 65)

    words = {s: set(ds[s][WORD_KEY]) for s in splits}
    pairs = {s: set(zip(ds[s][WORD_KEY], ds[s][GLOSS_KEY])) for s in splits}

    print(f"{'Total Samples':<25} | {len(ds['train']):<10} | {len(ds['validation']):<10} | {len(ds['test']):<10}")
    print(f"{'Unique Words':<25} | {len(words['train']):<10} | {len(words['validation']):<10} | {len(words['test']):<10}")
    print(f"{'Unique Pairs (W+G)':<25} | {len(pairs['train']):<10} | {len(pairs['validation']):<10} | {len(pairs['test']):<10}")

    print("\n--- Overlap Analysis (Leakage) ---")

    def print_overlap(label, set_dict):
        tr_val = len(set_dict['train'] & set_dict['validation'])
        tr_test = len(set_dict['train'] & set_dict['test'])
        print(f"{label:<25} | Train ∩ Val: {tr_val:<5} | Train ∩ Test: {tr_test}")

    print_overlap("Word Overlap", words)
    print_overlap("Full Pair Overlap", pairs)

    print("\n--- Data Integrity Insights ---")

    for s in splits:
        dupes = len(ds[s]) - len(pairs[s])
        if dupes > 0:
            print(f"{s.capitalize()}: Found {dupes} exact duplicate (Word+Gloss) rows.")

        if len(pairs[s]) > len(words[s]):
            ambiguity_count = len(pairs[s]) - len(words[s])
            print(f"{s.capitalize()}: Has {ambiguity_count} cases where one word has multiple glosses.")

    if len(pairs['test']) > 0:
        leak_pct = (len(pairs['train'] & pairs['test']) / len(pairs['test'])) * 100
        leak_pct2 = (len(pairs['validation'] & pairs['test']) / len(pairs['test'])) * 100
        print(f"\nCritical Leakage: {leak_pct:.2f}% of Test pairs are present in Training.")

In [ ]:
def build_reverse_dictionary_dataset():
  ds1, ds2 = load_data()

  # Normalize ds1
  for split in SPLITS:
      ds1[split] = ds1[split].remove_columns(COLUMNS_TO_REMOVE)
      ds1[split] = preprocess_dataset(ds1[split])
      ds1[split] = deduplicate(ds1[split])

  # Normalize ds2
  ds2 = ds2.rename_column("definition", "gloss")
  ds2 = preprocess_dataset(ds2)
  ds2 = deduplicate(ds2)

  # Remove ds2 entries already in ds1
  ds2 = remove_dublicates(ds1, ds2)

  # Merge
  dataset_full = merge_datasets([ds1["train"], ds1["test"], ds1["validation"], ds2])
  dataset_full = deduplicate(dataset_full)

  dataset_full = dataset_full.shuffle(SEED)

  train_test = dataset_full.train_test_split(test_size=0.2, seed=42)
  test_val = train_test["test"].train_test_split(test_size=0.5, seed=42)

  validation_ds = ds1["validation"]
  test_ds = ds1["test"]

  return {
      "train": train_test["train"],
      "validation": test_val["train"],
      "test": test_val["test"],
    }


In [ ]:
from datasets import DatasetDict
def save_dataset(dataset, path):
  final_ds = DatasetDict(dataset)

  final_ds.push_to_hub(path, private=True)

In [ ]:
dataset = build_reverse_dictionary_dataset()

Empty examples: 0
Empty examples: 0
Empty examples: 0
Empty examples: 0


In [ ]:
analyze_dataset_integrity(dataset)

Metric                    | Train      | Val        | Test      
-----------------------------------------------------------------
Total Samples             | 76265      | 9533       | 9534      
Unique Words              | 35310      | 7201       | 7205      
Unique Pairs (W+G)        | 76265      | 9533       | 9534      

--- Overlap Analysis (Leakage) ---
Word Overlap              | Train ∩ Val: 4060  | Train ∩ Test: 4072
Full Pair Overlap         | Train ∩ Val: 0     | Train ∩ Test: 0

--- Data Integrity Insights ---
Train: Has 40955 cases where one word has multiple glosses.
Validation: Has 2332 cases where one word has multiple glosses.
Test: Has 2329 cases where one word has multiple glosses.

Critical Leakage: 0.00% of Test pairs are present in Training.


In [ ]:
dataset

{'train': Dataset({
     features: ['word', 'gloss'],
     num_rows: 76265
 }),
 'validation': Dataset({
     features: ['word', 'gloss'],
     num_rows: 9533
 }),
 'test': Dataset({
     features: ['word', 'gloss'],
     num_rows: 9534
 })}

In [ ]:
save_dataset(dataset, CLEAN_DATA_HF_PATH)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/77 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 5.73MB / 5.73MB            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########|  722kB /  722kB            

README.md:   0%|          | 0.00/569 [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


## Model Experiments

In [2]:
from datasets import load_dataset, concatenate_datasets

dataset = load_dataset(CLEAN_DATA_HF_PATH)
train_ds = dataset['train']
val_ds = dataset['validation']
test_ds = dataset['test']

print(f"Train examples: {len(train_ds)}, Validation: {len(val_ds)}, Test: {len(test_ds)}")

Train examples: 76265, Validation: 9533, Test: 9534


In [3]:
from collections import defaultdict

count = defaultdict(int)
for ex in train_ds:
  count[len(ex[WORD].split())] += 1
print(count)

defaultdict(<class 'int'>, {1: 56142, 2: 15476, 3: 3638, 4: 807, 6: 36, 5: 153, 8: 3, 7: 8, 13: 1, 9: 1})


**TF-IDF**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 1),
    min_df=2,
    max_df=0.95,
    max_features=30000,
    sublinear_tf=True,
    dtype=np.float32
)

X_train = vectorizer.fit_transform(train_ds["gloss"])
train_words = list(train_ds["word"])

X_test = vectorizer.transform(test_ds["gloss"])
test_words = list(test_ds["word"])

In [ ]:
def retrieve_top_k(test_vectors, train_vectors, train_words, k=5):
    similarities = cosine_similarity(test_vectors, train_vectors)
    top_k_indices = np.argsort(-similarities, axis=1)[:, :k]
    return [[train_words[i] for i in idxs] for idxs in top_k_indices]

def evaluate(predictions, gold_words):
    top1 = sum(gold_words[i] == preds[0] for i, preds in enumerate(predictions)) / len(gold_words)
    top5 = sum(gold_words[i] in preds[:5] for i, preds in enumerate(predictions)) / len(gold_words)

    mrr = 0
    for i, preds in enumerate(predictions):
        if gold_words[i] in preds:
            mrr += 1 / (preds.index(gold_words[i]) + 1)
    mrr /= len(gold_words)

    return {"Top-1": top1, "Top-5": top5, "MRR": mrr}

In [ ]:
preds = retrieve_top_k(X_test, X_train, train_words, k=5)
metrics = evaluate(preds, test_words)
print(metrics)

**FAISS**

In [4]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 23.5 MB/s eta 0:00:00


In [3]:
import faiss

def build_faiss_index(vectors, save_path=None):
    """
    Creates a FAISS index for Cosine Similarity.
    """
    dims = vectors.shape[1]

    faiss.normalize_L2(vectors)

    index = faiss.IndexFlatIP(dims)

    index.add(vectors)

    if save_path:
        faiss.write_index(index, save_path)

    return index

**Static Embeddings**

In [ ]:
!pip install fasttext

In [ ]:
!wget https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.ar.300.bin.gz
!gunzip cc.ar.300.bin.gz

In [4]:
def get_top_k_matches(query_text, model, index, k=5):

    query_vec = model.get_sentence_vector(query_text).reshape(1, -1).astype('float32')
    faiss.normalize_L2(query_vec)
    scores, indices = index.search(query_vec, k)
    return scores[0], indices[0]

In [5]:
def vectorize_text_list(text_list, model):

    vectors = [model.get_sentence_vector(str(text)) for text in text_list]
    return np.array(vectors).astype('float32')

In [6]:
import fasttext
import numpy as np

model = fasttext.load_model('cc.ar.300.bin')

train_vectors = vectorize_text_list(train_ds["gloss"], model)
test_vectors = vectorize_text_list(test_ds["gloss"], model)

index = build_faiss_index(train_vectors)

print(f"Starting evaluation on {len(test_ds['word'])} samples...")

top1_count = 0
top5_count = 0
mrr_sum = 0

for i, query in enumerate(test_ds["gloss"],):
    scores, indices = get_top_k_matches(query, model, index, k=5)

    predicted_words = [train_ds[int(idx)]['word'] for idx in indices]
    correct_word = test_ds['word'][i]

    if correct_word == predicted_words[0]:
        top1_count += 1

    if correct_word in predicted_words:
        top5_count += 1
        rank = predicted_words.index(correct_word) + 1
        mrr_sum += (1.0 / rank)

print(f"Top-1 Accuracy: {top1_count / len(test_ds["gloss"],):.4f}")
print(f"Top-5 Accuracy: {top5_count / len(test_ds["gloss"],):.4f}")
print(f"MRR: {mrr_sum / len(test_ds["gloss"],):.4f}")

Starting evaluation on 9534 samples...
Top-1 Accuracy: 0.1504
Top-5 Accuracy: 0.2312
MRR: 0.1817


**Dynamic Embeddings**

**LLMs**